In [7]:
!pip install qiskit qiskit-aer qiskit-algorithms qiskit-optimization

In [8]:
import numpy as np
from qiskit_optimization import QuadraticProgram

TABLES = ['customer', 'orders', 'lineitem']
CARDS = {
    'customer': 150_000,
    'orders':   1_500_000,
    'lineitem': 6_001_215,
}
JOIN_EDGES = {
    frozenset(['customer', 'orders']):   1/150_000,
    frozenset(['orders',   'lineitem']): 1/1_500_000,
}
N = len(TABLES)

def pair_cost(i, j):
    t1, t2 = TABLES[i], TABLES[j]
    edge = frozenset([t1, t2])
    if edge in JOIN_EDGES:
        sel = JOIN_EDGES[edge]
        cost = CARDS[t1] * CARDS[t2] * sel
        return np.log10(max(cost, 1.0))
    return 15.0

def build_qubo(penalty=200.0):
    qp = QuadraticProgram('tpch_3table')
    for i in range(N):
        for p in range(N):
            qp.binary_var(name=f'x_{i}_{p}')
    linear, quadratic = {}, {}
    for i in range(N):
        for p in range(N):
            key = f'x_{i}_{p}'
            linear[key] = linear.get(key, 0) - 2 * penalty
        for p1 in range(N):
            for p2 in range(p1 + 1, N):
                key = (f'x_{i}_{p1}', f'x_{i}_{p2}')
                quadratic[key] = quadratic.get(key, 0) + 2 * penalty
    for p in range(N):
        for i in range(N):
            key = f'x_{i}_{p}'
            linear[key] = linear.get(key, 0) - 2 * penalty
        for i1 in range(N):
            for i2 in range(i1 + 1, N):
                key = (f'x_{i1}_{p}', f'x_{i2}_{p}')
                quadratic[key] = quadratic.get(key, 0) + 2 * penalty
    for p in range(N - 1):
        for i in range(N):
            for j in range(N):
                if i != j:
                    key = (f'x_{i}_{p}', f'x_{j}_{p+1}')
                    quadratic[key] = quadratic.get(key, 0) + pair_cost(i, j)
    qp.minimize(linear=linear, quadratic=quadratic, constant=N*2*penalty)
    return qp

print(f"QUBO ready: {N} tables, {N*N} qubits")

QUBO ready: 3 tables, 9 qubits


In [9]:
import time
from qiskit_algorithms import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler
from qiskit_optimization.algorithms import MinimumEigenOptimizer

def decode(x):
    order = [None] * N
    for i in range(N):
        for p in range(N):
            if round(x[i*N + p]) == 1:
                order[p] = TABLES[i]
    return order

qp = build_qubo(penalty=500.0)
print(f"Qubits: {qp.get_num_vars()}")

# Classical exact solution (baseline)
print("\n--- Classical exact ---")
start = time.perf_counter()
classical = MinimumEigenOptimizer(NumPyMinimumEigensolver()).solve(qp)
classical_time = time.perf_counter() - start
print(f"Time:           {classical_time*1000:.2f} ms")
print(f"Optimal order:  {decode(classical.x)}")
print(f"Optimal value:  {classical.fval:.4f}")

# QAOA p=1
print("\n--- QAOA p=1 ---")
np.random.seed(42)
qaoa = QAOA(sampler=StatevectorSampler(), optimizer=COBYLA(maxiter=50), reps=1)
start = time.perf_counter()
qaoa_result = MinimumEigenOptimizer(qaoa).solve(qp)
elapsed = time.perf_counter() - start
qaoa_order = decode(qaoa_result.x)
classical_order = decode(classical.x)
print(f"Time:           {elapsed:.1f} s")
print(f"QAOA order:     {qaoa_order}")
print(f"QAOA value:     {qaoa_result.fval:.4f}")
print(f"Match classical: {qaoa_order == classical_order}")

Qubits: 9

--- Classical exact ---
Time:           18.16 ms
Optimal order:  ['customer', 'orders', 'lineitem']
Optimal value:  -2987.0457

--- QAOA p=1 ---


/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


Time:           104.6 s
QAOA order:     ['customer', 'orders', 'lineitem']
QAOA value:     -2987.0457
Match classical: True


In [10]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit_optimization.converters import QuadraticProgramToQubo
import numpy as np

# Step 1: Extract optimal QAOA angles from the clean run we just did
optimal_params = qaoa_result.min_eigen_solver_result.optimal_parameters
optimal_values = list(optimal_params.values())
gamma_opt = optimal_values[0]
beta_opt = optimal_values[1]
print(f"Optimal angles from clean QAOA: gamma={gamma_opt:.4f}, beta={beta_opt:.4f}")

# Step 2: Convert QUBO to Ising Hamiltonian
converter = QuadraticProgramToQubo()
qubo_converted = converter.convert(qp)
ising_op, offset = qubo_converted.to_ising()
n_qubits = ising_op.num_qubits
print(f"Ising operator has {n_qubits} qubits")

# Step 3: Build QAOA circuit manually
def build_qaoa_circuit(gamma, beta, ising_op, n_qubits):
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))  # initial superposition

    # Cost layer: e^{-i*gamma*H_C}
    for pauli_term, coeff in zip(ising_op.paulis, ising_op.coeffs):
        z_indices = [i for i, p in enumerate(reversed(str(pauli_term))) if p == 'Z']
        if len(z_indices) == 1:
            qc.rz(2 * gamma * np.real(coeff), z_indices[0])
        elif len(z_indices) == 2:
            qc.cx(z_indices[0], z_indices[1])
            qc.rz(2 * gamma * np.real(coeff), z_indices[1])
            qc.cx(z_indices[0], z_indices[1])

    # Mixer layer: e^{-i*beta*B}
    for q in range(n_qubits):
        qc.rx(2 * beta, q)

    qc.measure_all()
    return qc

# Step 4: Build noise model
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.001, 1),
    ['u1', 'u2', 'u3', 'rx', 'rz', 'sx', 'x', 'h']
)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.01, 2),
    ['cx']
)
print("Noise model: 0.1% single-qubit, 1% two-qubit depolarizing")

# Step 5: Run with noise
qc = build_qaoa_circuit(gamma_opt, beta_opt, ising_op, n_qubits)
backend_noisy = AerSimulator(noise_model=noise_model)
transpiled = transpile(qc, backend_noisy)

print("\nRunning 8192 shots on noisy backend...")
job = backend_noisy.run(transpiled, shots=8192)
counts = job.result().get_counts()

# Step 6: Find most probable bitstring and decode
top_bitstring = max(counts, key=counts.get)
top_prob = counts[top_bitstring] / 8192
print(f"Most probable bitstring: {top_bitstring}")
print(f"Probability:             {top_prob:.4f}")

# Decode (bitstring is in reverse order in Qiskit)
x_noisy = [int(b) for b in reversed(top_bitstring[:N*N])]
noisy_order = decode(x_noisy)
print(f"Decoded order:           {noisy_order}")
print(f"Match optimal:           {noisy_order == classical_order}")

# Step 7: Also run noiseless for direct comparison
backend_clean = AerSimulator()
transpiled_clean = transpile(qc, backend_clean)
job_clean = backend_clean.run(transpiled_clean, shots=8192)
counts_clean = job_clean.result().get_counts()
top_clean = max(counts_clean, key=counts_clean.get)
prob_clean = counts_clean[top_clean] / 8192
print(f"\n--- Comparison ---")
print(f"Noiseless: most probable = {top_clean}, prob = {prob_clean:.4f}")
print(f"Noisy:     most probable = {top_bitstring}, prob = {top_prob:.4f}")
print(f"Probability degradation: {prob_clean - top_prob:.4f}")

Optimal angles from clean QAOA: gamma=3.3643, beta=2.6692
Ising operator has 9 qubits
Noise model: 0.1% single-qubit, 1% two-qubit depolarizing

Running 8192 shots on noisy backend...
Most probable bitstring: 000000000
Probability:             0.0089
Decoded order:           [None, None, None]
Match optimal:           False

--- Comparison ---
Noiseless: most probable = 000000000, prob = 0.0148
Noisy:     most probable = 000000000, prob = 0.0089
Probability degradation: 0.0059


In [12]:
!pip install qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.0/218.0 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 9.4 MB/s eta 0:00:00


In [ ]:
# Cell 5: Submit QAOA circuit to IBM Quantum hardware
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit import transpile, QuantumCircuit
import numpy as np

# === STEP 1: Enter your API token below ===
IBM_TOKEN = "PASTE YOUR TOKEN"

# Save credentials (only need to do this once)
QiskitRuntimeService.save_account(
    token=IBM_TOKEN,
    channel="ibm_quantum_platform",
    overwrite=True,
)
service = QiskitRuntimeService(channel="ibm_quantum_platform")
print("Connected to IBM Quantum Platform")

# === STEP 2: Pick the least busy real backend ===
backends = service.backends(simulator=False, operational=True)
print(f"\nAvailable real quantum backends: {len(backends)}")
for b in backends[:5]:
    print(f"  {b.name}: {b.num_qubits} qubits, queue={b.status().pending_jobs}")

backend = service.least_busy(simulator=False, operational=True, min_num_qubits=n_qubits)
print(f"\nSelected: {backend.name} ({backend.num_qubits} qubits)")
print(f"Queue position: {backend.status().pending_jobs} jobs ahead")

# === STEP 3: Use the QAOA circuit we already built ===
# qc is the manual QAOA circuit from Cell 4 (with optimal gamma, beta)
# Transpile for this specific backend
# Manual transpile to IBM's native gate set (bypasses backend plugin)
qc_for_hw = transpile(
    qc,
    basis_gates=['rz', 'sx', 'x', 'cz', 'id'],
    optimization_level=1,
    coupling_map=backend.coupling_map,
)
print(f"\nTranspiled circuit depth: {qc_for_hw.depth()}")
print(f"Transpiled gate count:    {qc_for_hw.size()}")

# === STEP 4: Submit ONE job, 4096 shots ===
sampler = SamplerV2(mode=backend)
sampler.options.default_shots = 4096

job = sampler.run([qc_for_hw])
print(f"\n=== JOB SUBMITTED ===")
print(f"Job ID: {job.job_id()}")
print(f"Status: {job.status()}")
print(f"\nKeep this Job ID safe. You can retrieve results later from:")
print(f"https://quantum.cloud.ibm.com/jobs/{job.job_id()}")

qiskit_runtime_service.__init__:WARNING:2026-05-22 07:42:23,668: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: Kanhaiya Mehta. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-05-22 07:42:23,669: Loading instance: Kanhaiya Mehta, plan: open


Connected to IBM Quantum Platform

Available real quantum backends: 3
  ibm_kingston: 156 qubits, queue=0
  ibm_marrakesh: 156 qubits, queue=0
  ibm_fez: 156 qubits, queue=17


qiskit_runtime_service.backends:WARNING:2026-05-22 07:42:32,866: Loading instance: Kanhaiya Mehta, plan: open
qiskit_runtime_service.backends:WARNING:2026-05-22 07:42:34,454: Using instance: Kanhaiya Mehta, plan: open



Selected: ibm_kingston (156 qubits)
Queue position: 0 jobs ahead

Transpiled circuit depth: 263
Transpiled gate count:    595

=== JOB SUBMITTED ===
Job ID: d880hr2s46sc73f8pqtg
Status: QUEUED

Keep this Job ID safe. You can retrieve results later from:
https://quantum.cloud.ibm.com/jobs/d880hr2s46sc73f8pqtg


In [17]:
# Check IBM Quantum job status and retrieve results if ready
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(channel="ibm_quantum_platform")
JOB_ID = "d880hr2s46sc73f8pqtg"

job = service.job(JOB_ID)
print(f"Job ID:  {JOB_ID}")
print(f"Status:  {job.status()}")
print(f"Backend: {job.backend().name}")

if str(job.status()) == "DONE":
    print("\n=== RESULTS READY ===")
    result = job.result()
    pub_result = result[0]
    counts = pub_result.data.meas.get_counts()

    total_shots = sum(counts.values())
    top_bitstring = max(counts, key=counts.get)
    top_count = counts[top_bitstring]
    top_prob = top_count / total_shots

    print(f"Total shots:        {total_shots}")
    print(f"Most probable:      {top_bitstring}")
    print(f"Count:              {top_count}")
    print(f"Probability:        {top_prob:.4f}")

    # Decode the bitstring back to join order
    n = 9  # 9 qubits for 3 tables
    x_hw = [int(b) for b in reversed(top_bitstring[:n])]
    hw_order = decode(x_hw)
    print(f"Decoded join order: {hw_order}")
    print(f"Match classical:    {hw_order == classical_order}")

    # Show top 5 most probable outcomes
    print(f"\n=== TOP 5 OUTCOMES ===")
    sorted_counts = sorted(counts.items(), key=lambda x: -x[1])[:5]
    for bs, ct in sorted_counts:
        print(f"  {bs}: {ct:4d} shots ({ct/total_shots*100:.2f}%)")
else:
    print(f"\nJob is still {job.status()}. We will check again later.")
    print(f"For now, move on to writing the Results section.")

qiskit_runtime_service.__init__:WARNING:2026-05-22 07:45:28,688: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: Kanhaiya Mehta. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().


Job ID:  d880hr2s46sc73f8pqtg
Status:  DONE
Backend: ibm_kingston

=== RESULTS READY ===
Total shots:        4096
Most probable:      110011001
Count:              27
Probability:        0.0066
Decoded join order: ['orders', 'lineitem', 'lineitem']
Match classical:    False

=== TOP 5 OUTCOMES ===
  110011001:   27 shots (0.66%)
  101111000:   26 shots (0.63%)
  000000000:   25 shots (0.61%)
  011000011:   25 shots (0.61%)
  110000110:   24 shots (0.59%)
